# Notebook 1 — Alur PCV Klasik Adaptif (Preprocessing → Saliency & Segmentasi `if-else` → Ekstraksi Fitur → Bounding Box Manual)

Notebook Google Colab ini memvisualisasikan pipeline **Pengolahan Citra & Visi Komputer (PCV) Klasik** murni (*tanpa Deep Learning*) secara bertahap pada citra kendaraan, dari gambar mentah hingga menghasilkan **Bounding Box manual tanpa label kelas** yang tepat mengunci objek (tidak bocor menjadi 1 frame penuh).

### Alur Tahapan yang Divisualisasikan
1. **Input Citra** — Unggah langsung lewat Colab (`SOURCE_MODE = "upload"`), Google Drive (`"drive"`), atau lokal (`data/helper`).
2. **Analisis Kondisi Citra & Aturan `if-else` Adaptif** — Secara otomatis mendeteksi *bar letterbox hitam*, mengukur pencahayaan (`mean_gray`), proporsi langit silau (`bright_ratio`), dan keramaian tepi latar (`border_edge_density`) untuk mengatur preprocessing & strategi segmentasi.
3. **Preprocessing** — Deteksi area konten (abaikan *letterbox*), Grayscale, Gaussian Blur, CLAHE + Gamma Correction, dan perbandingan Histogram Intensitas.
4. **Segmentasi Klasik (Saliency + GrabCut Adaptif vs 3 Baseline Klasik)** — Membandingkan langsung:
   - `otsu` (*Otsu Thresholding*), `canny_morph` (*Canny Edge + Morfologi*), `hsv` (*HSV Saturation Masking*), dan
   - **`saliency_grabcut` (★ Utama)**: gabungan **CIELAB Border-Contrast Saliency** + **Spectral Residual Saliency (FFT)** + **GrabCut adaptif (`if-else` silau vs normal)** + Morfologi.
5. **Ekstraksi Fitur** — Mengubah objek hasil segmentasi menjadi angka & visual:
   - **Fitur Bentuk / Geometri** (`Area`, `Perimeter`, `x, y, w, h`, `Aspect Ratio`, `Extent`, `Solidity`, `Circularity`, `Hu Moments`)
   - **Fitur Gradien Bentuk (HOG)**
   - **Fitur Tekstur (GLCM)** & **Ringkasan Warna RGB/HSV**
6. **Hasil Akhir** — **Bounding Box Manual Tanpa Label** (`cv2.boundingRect`), ringkasan alur 5-panel siap tempel ke PowerPoint (PNG 300 DPI), dan evaluasi semua gambar uji.

## Cell 1 — Setup Library & Folder Output Presentasi

In [ ]:
import math
import zipfile
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from skimage.feature import graycomatrix, graycoprops, hog

try:
    from google.colab import drive, files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

try:
    display  # type: ignore[name-defined]
except NameError:
    display = print

OUT_DIR = Path("/content/outputs_presentasi/nb1_pcv_klasik") if IN_COLAB else Path("./outputs_presentasi/nb1_pcv_klasik")
OUT_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams["figure.dpi"] = 120
plt.rcParams["savefig.dpi"] = 300
plt.rcParams["axes.titlesize"] = 11
plt.rcParams["axes.titleweight"] = "bold"


def save_fig(fig, filename: str):
    path = OUT_DIR / filename
    fig.savefig(path, dpi=300, bbox_inches="tight", facecolor="white")
    print(f"Tersimpan: {path}")
    return path


print(f"Siap! Environment Colab={IN_COLAB} | OpenCV={cv2.__version__} | Output={OUT_DIR}")

## Cell 2 — Muat Gambar Uji (Upload Langsung, Google Drive, atau Lokal `data/helper`)

* Di Google Colab, biarkan `SOURCE_MODE = "upload"` untuk mengunggah gambar dari folder `data/helper` (atau gambar kendaraan lain).
* Ubah `ACTIVE_INDEX` (`0`, `1`, `2`, `3`) untuk memilih gambar utama yang dibedah detail pada Cell 3–8 (Cell 9 otomatis menjalankan semua gambar sekaligus).

In [ ]:
SOURCE_MODE = "upload"  # Pilihan: "upload" | "drive" | "local"
DRIVE_IMAGE_DIR = "/content/drive/MyDrive/ferqueue/helper"
LOCAL_IMAGE_DIR = "data/helper"
ACTIVE_INDEX = 0  # Indeks gambar utama yang dianalisis detail pada Cell 3-8

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
UPLOAD_DIR = OUT_DIR / "_inputs"
UPLOAD_DIR.mkdir(parents=True, exist_ok=True)

image_paths = []
if SOURCE_MODE == "upload" and IN_COLAB:
    print("Silakan pilih 1 atau lebih gambar kendaraan (misal 4 gambar dari folder data/helper)...")
    uploaded = colab_files.upload()
    for fname, content in uploaded.items():
        dst = UPLOAD_DIR / Path(fname).name
        dst.write_bytes(content)
        if dst.suffix.lower() in IMAGE_EXTS:
            image_paths.append(dst)
elif SOURCE_MODE == "drive" and IN_COLAB:
    drive.mount("/content/drive")
    src_dir = Path(DRIVE_IMAGE_DIR)
    image_paths = sorted(p for p in src_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)
else:
    src_dir = Path(LOCAL_IMAGE_DIR)
    if src_dir.is_dir():
        image_paths = sorted(p for p in src_dir.iterdir() if p.suffix.lower() in IMAGE_EXTS)

if not image_paths:
    raise FileNotFoundError(
        "Belum ada gambar yang dimuat. Unggah minimal 1 gambar JPG/PNG atau periksa path folder."
    )


def load_rgb(path: Path) -> np.ndarray:
    with Image.open(path) as img:
        return np.asarray(ImageOps.exif_transpose(img).convert("RGB"), dtype=np.uint8)


loaded_images = [(p.name, load_rgb(p)) for p in image_paths]
ACTIVE_INDEX = min(ACTIVE_INDEX, len(loaded_images) - 1)
active_name, img_rgb = loaded_images[ACTIVE_INDEX]

print(f"Total gambar dimuat: {len(loaded_images)}")
for idx, (name, arr) in enumerate(loaded_images):
    marker = " <-- AKTIF" if idx == ACTIVE_INDEX else ""
    print(f"  [{idx}] {name} | ukuran={arr.shape[1]}x{arr.shape[0]} px | dtype={arr.dtype}{marker}")

cols = min(len(loaded_images), 4)
rows = math.ceil(len(loaded_images) / cols)
fig, axes = plt.subplots(rows, cols, figsize=(4.2 * cols, 3.6 * rows))
axes = np.atleast_1d(axes).ravel()
for i, ax in enumerate(axes):
    if i < len(loaded_images):
        name, arr = loaded_images[i]
        ax.imshow(arr)
        tag = " [AKTIF]" if i == ACTIVE_INDEX else ""
        ax.set_title(f"[{i}] {name[:24]}{tag}\n{arr.shape[1]}x{arr.shape[0]} px", fontsize=9)
    ax.axis("off")
fig.suptitle("Daftar Gambar Uji yang Dimuat", fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig(fig, "00_daftar_gambar_uji.png")
plt.show()

## Cell 3 — Analisis Kondisi Citra & Aturan `if-else` Adaptif (Terkalibrasi pada 1.079 Gambar `data/inference`)

Sel ini secara otomatis membaca properti piksel gambar (bukan nama file) dan menentukan strategi pengolahan lewat **3 aturan `if-else`** yang ambang batasnya telah dikalibrasi secara kuantitatif pada folder `data/inference` (1.079 gambar):
1. **Deteksi Letterbox Hitam**: mendeteksi bar hitam padding atas/bawah (`mean < 14, std < 10`) — ditemukan pada 33% dataset — agar tidak merusak statistik maupun kontur.
2. **Aturan Pencahayaan (`mean_gray`)**:
   - `mean_gray < 85` (**Citra Gelap / Malam**) → aktifkan peningkatan kontras **LAB-CLAHE (`clip=2.2`) + Gamma (`0.78`)** sebelum GrabCut agar batas bodi kendaraan gelap tidak menyatu dengan bayangan/aspal.
   - `mean_gray > 180` (**Terang/Silau**) → `gamma=1.30`, `clahe_clip=1.5`.
   - Normal → `gamma=1.0`, `clahe_clip=2.0`.
3. **Aturan Proporsi Langit Silau (`bright_ratio = mean(gray > 230)`)**:
   - `if bright_ratio > 0.22` (**Silau / Langit Terang > 22%**) → strategi **`peak_saliency_roi`** (lokalisasi puncak saliency global + scan garis atap non-langit + *sky masking* + *fallback* otomatis bila mask < 3%).
   - `else` (**Kondisi Normal**) → strategi **`inner_rect`** dengan **`grabcut_margin = 0.05`** (margin 5% tepi frame sebagai *definite background*) + Morfologi (`open=5, close=11`).

> Ubah `AUTO_ADAPTIVE = False` jika ingin mencoba parameter manual sendiri di `MANUAL_PARAMS`.

In [ ]:
AUTO_ADAPTIVE = True  # True = gunakan aturan if-else terkalibrasi | False = gunakan MANUAL_PARAMS

MANUAL_PARAMS = {
    "gaussian_kernel": 5,
    "clahe_clip": 2.0,
    "clahe_grid": 8,
    "gamma": 1.0,
    "canny_low": 60,
    "canny_high": 160,
    "morph_close": 11,
    "morph_open": 5,
    "hsv_sat_thresh": 45,
    "grabcut_strategy": "inner_rect",  # Pilihan: "inner_rect" | "peak_saliency_roi"
    "grabcut_margin": 0.05,
    "grabcut_iter": 5,
    "grabcut_max_dim": 320,
    "enhance_dark_rgb": True,
    "peak_fallback": True,
    "saliency_k_bg": 6,
    "saliency_peak_ratio": 0.68,
    "sky_gray_thresh": 235,
}


def detect_black_letterbox(rgb: np.ndarray) -> tuple:
    """Deteksi area konten aktif dengan mengabaikan bar letterbox hitam murni di tepi gambar."""
    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    h, w = gray.shape
    bad_r = (gray.mean(axis=1) < 14) & (gray.std(axis=1) < 10)
    bad_c = (gray.mean(axis=0) < 14) & (gray.std(axis=0) < 10)
    y1, y2, x1, x2 = 0, h, 0, w
    while y1 < h // 3 and bad_r[y1]:
        y1 += 1
    while y2 > 2 * h // 3 and bad_r[y2 - 1]:
        y2 -= 1
    while x1 < w // 3 and bad_c[x1]:
        x1 += 1
    while x2 > 2 * w // 3 and bad_c[x2 - 1]:
        x2 -= 1
    return y1, y2, x1, x2


def analyze_image_and_choose_rules(rgb: np.ndarray, auto: bool = True, manual_cfg: dict = None) -> dict:
    """Hitung statistik piksel area konten aktif dan tentukan parameter lewat aturan if-else terkalibrasi."""
    y1, y2, x1, x2 = detect_black_letterbox(rgb)
    crop = rgb[y1:y2, x1:x2]
    h, w = crop.shape[:2]
    gray = cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY)
    hsv = cv2.cvtColor(crop, cv2.COLOR_RGB2HSV)

    border_mask = np.ones((h, w), dtype=bool)
    by, bx = max(1, int(0.12 * h)), max(1, int(0.12 * w))
    border_mask[by : h - by, bx : w - bx] = False

    mean_gray = float(gray.mean())
    median_gray = float(np.median(gray))
    bright_ratio = float(np.mean(gray > 230))
    border_sat = float(hsv[:, :, 1][border_mask].mean())
    quick_edges = cv2.Canny(cv2.GaussianBlur(gray, (5, 5), 0), 60, 150)
    border_edge_density = float(np.mean(quick_edges[border_mask] > 0))
    has_letterbox = (y1 > 2) or (y2 < rgb.shape[0] - 2) or (x1 > 2) or (x2 < rgb.shape[1] - 2)

    stats = {
        "content_roi": f"y={y1}:{y2}, x={x1}:{x2}",
        "letterbox": "Ya" if has_letterbox else "Tidak",
        "mean_gray": round(mean_gray, 1),
        "bright_ratio_%": round(bright_ratio * 100, 1),
        "border_edge": round(border_edge_density, 4),
        "border_sat": round(border_sat, 1),
    }

    if not auto:
        cfg = dict(manual_cfg or MANUAL_PARAMS)
        cfg["roi"] = (y1, y2, x1, x2)
        cfg["mean_gray"] = mean_gray
        cfg["stats"] = stats
        cfg["conditions"] = ["Mode Manual (aturan if-else dinonaktifkan)"]
        return cfg

    conditions = []
    if has_letterbox:
        conditions.append(f"Letterbox hitam terdeteksi -> fokus area konten y={y1}:{y2}, x={x1}:{x2}")
    else:
        conditions.append("Tanpa letterbox -> seluruh frame diproses")

    # ATURAN 1: Pencahayaan -> GAMMA, CLAHE, & Dark-RGB Enhancement sebelum GrabCut
    enhance_dark_rgb = False
    if mean_gray < 85:
        gamma, clahe_clip = 0.78, 2.2
        enhance_dark_rgb = True
        conditions.append(f"Pencahayaan Gelap (mean={mean_gray:.1f}<85) -> gamma=0.78, LAB-CLAHE=2.2 (aktif utk GrabCut)")
    elif mean_gray > 180:
        gamma, clahe_clip = 1.30, 1.5
        conditions.append(f"Pencahayaan Terang/Silau (mean={mean_gray:.1f}>180) -> gamma=1.30, clahe=1.5")
    else:
        gamma, clahe_clip = 1.0, 2.0
        conditions.append(f"Pencahayaan Normal (mean={mean_gray:.1f}) -> gamma=1.0, clahe=2.0")

    # ATURAN 2: Keramaian Tepi -> Gaussian Blur & Canny (untuk baseline pembanding)
    if border_edge_density > 0.08:
        gaussian_k = 7
        canny_low = int(np.clip(0.75 * median_gray, 70, 130))
        canny_high = int(np.clip(1.80 * median_gray, 170, 240))
        conditions.append(f"Background Ramai (edge_tepi={border_edge_density:.3f}>0.08) -> blur=7, Canny={canny_low}/{canny_high}")
    else:
        gaussian_k = 5
        canny_low = int(np.clip(0.60 * median_gray, 35, 90))
        canny_high = int(np.clip(1.35 * median_gray, 100, 200))
        conditions.append(f"Background Sedang/Polos (edge_tepi={border_edge_density:.3f}<=0.08) -> blur=5, Canny={canny_low}/{canny_high}")

    # ATURAN 3: Proporsi Langit Silau (>22% hasil kalibrasi 1.079 gambar) -> Strategi Saliency + GrabCut
    if bright_ratio > 0.22:
        gc_strategy = "peak_saliency_roi"
        conditions.append(
            f"Langit Terang/Silau Luas ({bright_ratio*100:.1f}% > 22%) -> Peak-Saliency ROI GrabCut + Sky Masking"
        )
    else:
        gc_strategy = "inner_rect"
        conditions.append(
            f"Kondisi Normal (silau={bright_ratio*100:.1f}% <= 22%) -> Inner-Frame GrabCut (margin=5%) + Morfologi"
        )

    return {
        "roi": (y1, y2, x1, x2),
        "mean_gray": mean_gray,
        "gaussian_kernel": gaussian_k,
        "clahe_clip": clahe_clip,
        "clahe_grid": 8,
        "gamma": gamma,
        "canny_low": canny_low,
        "canny_high": canny_high,
        "morph_close": 11,
        "morph_open": 5,
        "hsv_sat_thresh": int(np.clip(border_sat + 25, 35, 95)),
        "grabcut_strategy": gc_strategy,
        "grabcut_margin": 0.05,
        "grabcut_iter": 5,
        "grabcut_max_dim": 320,
        "enhance_dark_rgb": enhance_dark_rgb,
        "peak_fallback": True,
        "saliency_k_bg": 6,
        "saliency_peak_ratio": 0.68,
        "sky_gray_thresh": 235,
        "stats": stats,
        "conditions": conditions,
    }


rule_cfg = analyze_image_and_choose_rules(img_rgb, auto=AUTO_ADAPTIVE, manual_cfg=MANUAL_PARAMS)

print(f"=== HASIL ANALISIS KONDISI CITRA ({active_name}) ===")
for line in rule_cfg["conditions"]:
    print("  •", line)

display(pd.DataFrame([rule_cfg["stats"]]))
df_rules = pd.DataFrame([
    {"Parameter": k, "Nilai_Terpilih": rule_cfg[k]}
    for k in (
        "grabcut_strategy", "gaussian_kernel", "clahe_clip", "gamma",
        "canny_low", "canny_high", "morph_close", "morph_open", "hsv_sat_thresh",
    )
])
display(df_rules)

## Cell 4 — Tahap 1: Preprocessing Citra (Letterbox ROI, Grayscale, Gaussian Blur, CLAHE & Histogram)

Menjalankan preprocessing pada area konten aktif sesuai parameter hasil aturan `if-else`:
1. **Deteksi Area Konten** (garis batas putus-putus jika ada *letterbox* hitam)
2. **Grayscale** (1 channel intensitas `0–255`)
3. **Gaussian Blur** (meredam tekstur halus aspal/paving/daun)
4. **CLAHE + Gamma Correction** (menyeimbangkan kontras lokal) + **Histogram Intensitas**.

In [ ]:
def preprocess_classic(rgb: np.ndarray, cfg: dict) -> dict:
    ksize = int(cfg["gaussian_kernel"])
    clip = float(cfg["clahe_clip"])
    grid = int(cfg.get("clahe_grid", 8))
    gamma = float(cfg["gamma"])

    gray = cv2.cvtColor(rgb, cv2.COLOR_RGB2GRAY)
    blurred = cv2.GaussianBlur(gray, (ksize, ksize), 0)
    clahe = cv2.createCLAHE(clipLimit=clip, tileGridSize=(grid, grid))
    enhanced = clahe.apply(blurred)
    if gamma != 1.0:
        lut = np.round(255.0 * (np.arange(256) / 255.0) ** gamma).astype(np.uint8)
        enhanced = cv2.LUT(enhanced, lut)

    rgb_blur = cv2.GaussianBlur(rgb, (ksize, ksize), 0)
    lab = cv2.cvtColor(rgb_blur, cv2.COLOR_RGB2LAB)
    lab[:, :, 0] = clahe.apply(lab[:, :, 0])
    rgb_enhanced = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)

    return {
        "rgb": rgb,
        "gray": gray,
        "blurred": blurred,
        "enhanced": enhanced,
        "rgb_enhanced": rgb_enhanced,
    }


prep = preprocess_classic(img_rgb, rule_cfg)
y1, y2, x1, x2 = rule_cfg["roi"]
roi_preview = img_rgb.copy()
cv2.rectangle(roi_preview, (x1 + 2, y1 + 2), (x2 - 3, y2 - 3), (0, 220, 255), 3)

fig = plt.figure(figsize=(16, 7.2))
gs = fig.add_gridspec(2, 4, height_ratios=[1.35, 1.0], hspace=0.35, wspace=0.18)

panels = [
    ("1. Input + Area Konten (Cyan)", roi_preview, None),
    ("2. Grayscale (1 Channel)", prep["gray"], "gray"),
    (f"3. Gaussian Blur ({rule_cfg['gaussian_kernel']}x{rule_cfg['gaussian_kernel']})", prep["blurred"], "gray"),
    (f"4. CLAHE (clip={rule_cfg['clahe_clip']}, γ={rule_cfg['gamma']})", prep["enhanced"], "gray"),
]
for i, (title, im, cmap) in enumerate(panels):
    ax = fig.add_subplot(gs[0, i])
    ax.imshow(im, cmap=cmap)
    ax.set_title(title)
    ax.axis("off")

ax_h1 = fig.add_subplot(gs[1, 0:2])
ax_h1.hist(prep["gray"][y1:y2, x1:x2].ravel(), bins=64, range=(0, 256), color="#4c72b0", alpha=0.85)
ax_h1.set_title("Histogram Intensitas Sebelum CLAHE (Area Konten)")
ax_h1.set_xlabel("Nilai Piksel (0 - 255)")
ax_h1.set_ylabel("Jumlah Piksel")
ax_h1.grid(alpha=0.25)

ax_h2 = fig.add_subplot(gs[1, 2:4])
ax_h2.hist(prep["enhanced"][y1:y2, x1:x2].ravel(), bins=64, range=(0, 256), color="#dd8452", alpha=0.85)
ax_h2.set_title("Histogram Intensitas Sesudah CLAHE + Gamma")
ax_h2.set_xlabel("Nilai Piksel (0 - 255)")
ax_h2.set_ylabel("Jumlah Piksel")
ax_h2.grid(alpha=0.25)

fig.suptitle(f"Tahap 1: Preprocessing PCV Klasik — {active_name}", fontsize=14, fontweight="bold")
save_fig(fig, "01_preprocessing.png")
plt.show()

## Cell 5 — Tahap 2: Segmentasi Klasik (Saliency + GrabCut Adaptif vs 3 Baseline Klasik)

Di sel ini kita membandingkan **4 metode segmentasi PCV klasik murni (tanpa Deep Learning)** pada gambar yang sama:
1. **Otsu Thresholding** — ambang intensitas global (sering gagal jika bodi kendaraan punya warna terang & gelap sekaligus).
2. **Canny Edge + Morfologi** — deteksi tepi + penutupan celah (rentan terhadap tekstur pohon/paving di background).
3. **HSV Saturation Mask** — memisahkan objek berdasarkan kepekatan warna (gagal pada mobil putih/silver/hitam).
4. **Saliency + GrabCut Adaptif (`if-else`) ★ Utama**:
   - Menghitung **Peta Saliency Klasik** = **CIELAB Border-Contrast** *(jarak warna piksel terhadap klaster K-Means tepi frame)* + **Spectral Residual (FFT, Hou & Zhang 2007)**.
   - Memilih strategi GrabCut sesuai aturan `if-else` Cell 3 (`peak_saliency_roi` untuk langit silau/bangunan pinggir jalan, atau `inner_rect` untuk kondisi normal) lalu dirapikan dengan **Morfologi (Opening & Closing)**.

In [ ]:
def compute_classical_saliency(crop: np.ndarray, k_bg: int = 6) -> np.ndarray:
    """Saliency PCV klasik: CIELAB Border-Contrast + Spectral Residual (FFT) + Center Prior."""
    h, w = crop.shape[:2]
    blur = cv2.GaussianBlur(crop, (7, 7), 0)
    lab = cv2.cvtColor(blur, cv2.COLOR_RGB2LAB).astype(np.float32)

    by, bx = max(4, int(0.08 * h)), max(4, int(0.08 * w))
    border_pts = np.vstack([
        lab[:by, :, :].reshape(-1, 3),
        lab[h - by :, :, :].reshape(-1, 3),
        lab[:, :bx, :].reshape(-1, 3),
        lab[:, w - bx :, :].reshape(-1, 3),
    ])
    step = max(1, len(border_pts) // 2500)
    sample = border_pts[::step].astype(np.float32)
    cv2.setRNGSeed(42)
    _, _, centers = cv2.kmeans(
        sample, k_bg, None,
        (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 20, 1.0),
        3, cv2.KMEANS_PP_CENTERS,
    )
    dists = np.linalg.norm(lab.reshape(-1, 3)[:, None, :] - centers[None, :, :], axis=2)
    sal_lab = cv2.normalize(dists.min(axis=1).reshape(h, w), None, 0.0, 1.0, cv2.NORM_MINMAX)

    # Spectral Residual Saliency (Hou & Zhang, CVPR 2007)
    small = cv2.resize(cv2.cvtColor(crop, cv2.COLOR_RGB2GRAY), (64, 64)).astype(np.float32)
    fft = np.fft.fft2(small)
    log_amp = np.log(np.abs(fft) + 1e-8)
    residual = log_amp - cv2.blur(log_amp, (3, 3))
    sal_s = np.abs(np.fft.ifft2(np.exp(residual + 1j * np.angle(fft)))) ** 2
    sal_s = cv2.GaussianBlur(sal_s.astype(np.float32), (9, 9), 2.5)
    sal_spec = cv2.normalize(cv2.resize(sal_s, (w, h), interpolation=cv2.INTER_CUBIC), None, 0.0, 1.0, cv2.NORM_MINMAX)

    yy, xx = np.mgrid[0:h, 0:w]
    dx = (xx - w * 0.5) / (w * 0.5)
    dy = (yy - h * 0.5) / (h * 0.5)
    center_w = np.exp(-0.75 * (dx ** 2 + dy ** 2))

    comb = (0.60 * sal_lab + 0.40 * sal_spec) * (0.30 + 0.70 * center_w)
    sal_u8 = cv2.normalize(comb, None, 0, 255, cv2.NORM_MINMAX).astype(np.uint8)
    return cv2.GaussianBlur(sal_u8, (9, 9), 0)


def segment_saliency_grabcut(rgb: np.ndarray, cfg: dict) -> tuple:
    """Segmentasi utama PCV klasik adaptif (Saliency + GrabCut terkalibrasi + Morfologi)."""
    y1, y2, x1, x2 = cfg["roi"]
    crop = rgb[y1:y2, x1:x2]
    h, w = crop.shape[:2]

    # Jika citra gelap (mean_gray < 85), tingkatkan kontras LAB-CLAHE + Gamma sebelum GrabCut
    work_crop = crop
    if cfg.get("enhance_dark_rgb", True) and float(cfg.get("mean_gray", 128.0)) < 85.0:
        lab = cv2.cvtColor(work_crop, cv2.COLOR_RGB2LAB)
        lab[:, :, 0] = cv2.createCLAHE(clipLimit=2.2, tileGridSize=(8, 8)).apply(lab[:, :, 0])
        work_crop = cv2.cvtColor(lab, cv2.COLOR_LAB2RGB)
        lut = np.round(255.0 * (np.arange(256) / 255.0) ** 0.78).astype(np.uint8)
        work_crop = cv2.LUT(work_crop, lut)

    sal = compute_classical_saliency(work_crop, k_bg=int(cfg.get("saliency_k_bg", 6)))

    # Proses GrabCut pada skala maks 320px agar cepat & konsisten dengan kalibrasi 1.079 gambar
    scale = min(1.0, float(cfg.get("grabcut_max_dim", 320)) / max(h, w))
    sw, sh = max(32, int(w * scale)), max(32, int(h * scale))
    crop_s = cv2.resize(work_crop, (sw, sh), interpolation=cv2.INTER_AREA)
    sal_s = cv2.resize(sal, (sw, sh), interpolation=cv2.INTER_AREA)
    gray_s = cv2.cvtColor(crop_s, cv2.COLOR_RGB2GRAY)

    cv2.setRNGSeed(42)
    bgd = np.zeros((1, 65), np.float64)
    fgd = np.zeros((1, 65), np.float64)
    iters = int(cfg.get("grabcut_iter", 5))
    strategy = cfg.get("grabcut_strategy", "inner_rect")

    used_peak = False
    if strategy == "peak_saliency_roi":
        sal_inner = sal_s.copy()
        bx, by = max(4, int(0.08 * sw)), max(4, int(0.08 * sh))
        sal_inner[:, :bx] = 0
        sal_inner[:, sw - bx :] = 0
        sal_inner[:by, :] = 0
        sal_inner[sh - by :, :] = 0

        peak_ratio = float(cfg.get("saliency_peak_ratio", 0.68))
        peak_th = max(120, int(peak_ratio * sal_inner.max()))
        core = (sal_inner >= peak_th).astype(np.uint8) * 255
        core = cv2.morphologyEx(core, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (7, 7)))
        cnts_c, _ = cv2.findContours(core, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if cnts_c:
            def core_peak_score(c):
                m = np.zeros((sh, sw), dtype=np.uint8)
                cv2.drawContours(m, [c], -1, 255, cv2.FILLED)
                vals = sal_s[m > 0]
                return (float(vals.max()), float(vals.mean()))

            best_core = max(cnts_c, key=core_peak_score)
            cx, cy, cw, ch = cv2.boundingRect(best_core)

            est_w = max(cw, int(0.32 * sw))
            mid_x = cx + cw // 2
            rx1 = max(2, int(mid_x - 0.85 * est_w))
            rx2 = min(sw - 3, int(mid_x + 0.65 * est_w))

            sky_th = int(cfg.get("sky_gray_thresh", 235))
            non_sky_frac = np.mean(gray_s[:, rx1:rx2] < (sky_th - 3), axis=1)
            ry1 = cy
            sky_streak = 0
            for yy_idx in range(cy, 1, -1):
                if non_sky_frac[yy_idx] > 0.08:
                    ry1 = yy_idx
                    sky_streak = 0
                else:
                    sky_streak += 1
                    if sky_streak >= max(8, int(0.04 * sh)):
                        break
            ry1 = max(2, ry1 - int(0.04 * sh))
            ry2 = min(sh - 3, cy + ch + int(0.08 * sh))

            gc = np.full((sh, sw), cv2.GC_BGD, dtype=np.uint8)
            gc[ry1:ry2, rx1:rx2] = cv2.GC_PR_FGD
            gc[gray_s > sky_th] = cv2.GC_BGD
            core_fg = np.zeros((sh, sw), dtype=np.uint8)
            cv2.drawContours(core_fg, [best_core], -1, 255, cv2.FILLED)
            gc[(core_fg > 0) & (gc != cv2.GC_BGD)] = cv2.GC_FGD
            cv2.grabCut(crop_s, gc, None, bgd, fgd, iters, cv2.GC_INIT_WITH_MASK)

            fg_raw = np.isin(gc, (cv2.GC_FGD, cv2.GC_PR_FGD)).astype(np.uint8) * 255
            k_vert = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 25))
            fg_s = cv2.morphologyEx(fg_raw, cv2.MORPH_CLOSE, k_vert)
            used_peak = True
            # Fallback aman bila mask hasil peak ROI terlalu kecil (< 3%)
            if cfg.get("peak_fallback", True) and np.mean(fg_s > 0) < 0.03:
                used_peak = False

    if not used_peak:
        margin = float(cfg.get("grabcut_margin", 0.05))
        mx, my = max(2, int(margin * sw)), max(2, int(margin * sh))
        gc = np.zeros((sh, sw), dtype=np.uint8)
        cv2.grabCut(crop_s, gc, (mx, my, sw - 2 * mx, sh - 2 * my), bgd, fgd, iters, cv2.GC_INIT_WITH_RECT)
        fg_s = np.isin(gc, (cv2.GC_FGD, cv2.GC_PR_FGD)).astype(np.uint8) * 255

    ok, ck = int(cfg["morph_open"]), int(cfg["morph_close"])
    k_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ok, ok))
    k_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (ck, ck))
    fg_s = cv2.morphologyEx(cv2.morphologyEx(fg_s, cv2.MORPH_CLOSE, k_close), cv2.MORPH_OPEN, k_open)

    fg_crop = cv2.resize(fg_s, (w, h), interpolation=cv2.INTER_NEAREST)
    cnts, _ = cv2.findContours(fg_crop, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    best_crop = np.zeros_like(fg_crop)
    if cnts:
        largest = max(cnts, key=cv2.contourArea)
        cv2.drawContours(best_crop, [largest], -1, 255, cv2.FILLED)

    full_mask = np.zeros(rgb.shape[:2], dtype=np.uint8)
    full_mask[y1:y2, x1:x2] = best_crop
    full_sal = np.zeros(rgb.shape[:2], dtype=np.uint8)
    full_sal[y1:y2, x1:x2] = sal
    return full_sal, full_mask


def auto_orient_foreground(binary_mask: np.ndarray) -> np.ndarray:
    border = np.concatenate([binary_mask[0, :], binary_mask[-1, :], binary_mask[:, 0], binary_mask[:, -1]])
    return cv2.bitwise_not(binary_mask) if np.mean(border > 0) > 0.5 else binary_mask


def clean_roi_mask(mask_crop: np.ndarray, open_k: int, close_k: int) -> np.ndarray:
    ko = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_k, open_k))
    kc = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_k, close_k))
    m = cv2.morphologyEx(cv2.morphologyEx(mask_crop, cv2.MORPH_CLOSE, kc), cv2.MORPH_OPEN, ko)
    cnts, _ = cv2.findContours(m, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    out = np.zeros_like(m)
    min_a = 0.015 * m.shape[0] * m.shape[1]
    for c in cnts:
        if cv2.contourArea(c) >= min_a:
            cv2.drawContours(out, [c], -1, 255, cv2.FILLED)
    return out


def run_all_classic_segmentations(prep_dict: dict, cfg: dict) -> tuple:
    rgb = prep_dict["rgb"]
    y1, y2, x1, x2 = cfg["roi"]
    enh_c = prep_dict["enhanced"][y1:y2, x1:x2]
    rgb_enh_c = prep_dict["rgb_enhanced"][y1:y2, x1:x2]
    ok, ck = int(cfg["morph_open"]), int(cfg["morph_close"])

    def put_back(crop_img):
        full = np.zeros(rgb.shape[:2], dtype=np.uint8)
        full[y1:y2, x1:x2] = crop_img
        return full

    # 1. Otsu Thresholding (baseline)
    _, otsu_raw = cv2.threshold(enh_c, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    otsu_raw = auto_orient_foreground(otsu_raw)
    otsu_mask = clean_roi_mask(otsu_raw, ok, ck)

    # 2. Canny Edge + Morfologi (baseline)
    edges = cv2.Canny(enh_c, int(cfg["canny_low"]), int(cfg["canny_high"]))
    dil = cv2.dilate(edges, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)), iterations=2)
    canny_mask = clean_roi_mask(dil, ok, ck)

    # 3. HSV Saturation (baseline)
    hsv_th = int(cfg["hsv_sat_thresh"])
    sat_raw = (cv2.cvtColor(rgb_enh_c, cv2.COLOR_RGB2HSV)[:, :, 1] >= hsv_th).astype(np.uint8) * 255
    hsv_mask = clean_roi_mask(sat_raw, ok, ck)

    # 4. Saliency + GrabCut Adaptif (metode utama pilihan if-else)
    sal_map, sal_gc_mask = segment_saliency_grabcut(rgb, cfg)

    results = {
        "otsu": {"label": "Otsu Thresholding", "raw": put_back(otsu_raw), "mask": put_back(otsu_mask), "raw_cmap": "gray"},
        "canny_morph": {"label": "Canny + Morfologi", "raw": put_back(edges), "mask": put_back(canny_mask), "raw_cmap": "gray"},
        "hsv": {"label": f"HSV Saturation (th={hsv_th})", "raw": put_back(sat_raw), "mask": put_back(hsv_mask), "raw_cmap": "gray"},
        "saliency_grabcut": {
            "label": f"Saliency + GrabCut ({cfg['grabcut_strategy']})",
            "raw": sal_map,
            "mask": sal_gc_mask,
            "raw_cmap": "inferno",
        },
    }
    return results, "saliency_grabcut"


def make_overlay(rgb: np.ndarray, mask: np.ndarray, color=(40, 220, 90), alpha=0.45) -> np.ndarray:
    overlay = rgb.copy().astype(np.float32)
    tint = np.zeros_like(overlay)
    tint[:, :] = color
    m = mask > 0
    overlay[m] = (1 - alpha) * overlay[m] + alpha * tint[m]
    return overlay.astype(np.uint8)


seg_results, auto_selected_key = run_all_classic_segmentations(prep, rule_cfg)

display_keys = ["otsu", "canny_morph", "hsv", "saliency_grabcut"]
fig, axes = plt.subplots(3, 4, figsize=(16.5, 10.5))
row_titles = ["Peta Awal / Saliency", "Mask Biner Bersih", "Overlay pada Citra Asli"]

for col, key in enumerate(display_keys):
    res = seg_results[key]
    cov = 100.0 * np.mean(res["mask"] > 0)
    star = " ★ DIPILIH" if key == auto_selected_key else ""
    axes[0, col].imshow(res["raw"], cmap=res["raw_cmap"])
    axes[0, col].set_title(f"{res['label']}{star}\n(key='{key}')", fontsize=9)
    axes[1, col].imshow(res["mask"], cmap="gray")
    axes[1, col].set_title(f"Mask Bersih (area={cov:.1f}%)", fontsize=9.5)
    axes[2, col].imshow(make_overlay(img_rgb, res["mask"]))
    axes[2, col].set_title("Overlay Foreground", fontsize=9.5)
    for r in range(3):
        axes[r, col].axis("off")

for r, rt in enumerate(row_titles):
    axes[r, 0].text(
        -0.08, 0.5, rt, transform=axes[r, 0].transAxes,
        rotation=90, va="center", ha="right", fontsize=11, fontweight="bold",
    )

fig.suptitle(f"Tahap 2: Segmentasi Klasik vs Saliency + GrabCut Adaptif — {active_name[:36]}", fontsize=13.5, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
save_fig(fig, "02_perbandingan_segmentasi.png")
plt.show()

## Cell 6 — Penyempurnaan Mask Terpilih, Kontur & Isolasi Objek

Secara default, sel ini memakai hasil terbaik **`saliency_grabcut`** (`OVERRIDE_METHOD = None`).
Jika ingin memaksa melihat metode lain (`"otsu"`, `"canny_morph"`, `"hsv"`), cukup ubah `OVERRIDE_METHOD`.

In [ ]:
OVERRIDE_METHOD = None   # None = pakai 'saliency_grabcut' | atau isi: "otsu", "canny_morph", "hsv"
MIN_AREA_RATIO = 0.02    # Abaikan kontur noise < 2% luas gambar
MAX_AREA_RATIO = 0.85    # Abaikan kontur yang menelan hampir seluruh frame (> 85%)
KEEP_TOP_K_OBJECTS = 1   # Jumlah maksimal objek yang diambil

SELECTED_METHOD = OVERRIDE_METHOD if OVERRIDE_METHOD in seg_results else auto_selected_key


def extract_valid_contours(mask: np.ndarray, min_ratio=0.02, max_ratio=0.85, top_k=1):
    h, w = mask.shape
    img_area = float(h * w)
    contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    valid = []
    for cnt in contours:
        area = cv2.contourArea(cnt)
        ratio = area / img_area
        if min_ratio <= ratio <= max_ratio:
            valid.append((area, cnt))
    valid.sort(key=lambda item: item[0], reverse=True)
    return [cnt for _, cnt in valid[:top_k]]


chosen_raw = seg_results[SELECTED_METHOD]["raw"]
chosen_cmap = seg_results[SELECTED_METHOD]["raw_cmap"]
chosen_mask = seg_results[SELECTED_METHOD]["mask"]
chosen_label = seg_results[SELECTED_METHOD]["label"]
valid_contours = extract_valid_contours(chosen_mask, MIN_AREA_RATIO, MAX_AREA_RATIO, KEEP_TOP_K_OBJECTS)

if not valid_contours:
    print("PERINGATAN: Tidak ada kontur lolos filter area. Mengambil kontur terbesar yang tersedia.")
    all_cnts, _ = cv2.findContours(chosen_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if all_cnts:
        valid_contours = [max(all_cnts, key=cv2.contourArea)]

selected_mask = np.zeros_like(chosen_mask)
cv2.drawContours(selected_mask, valid_contours, -1, 255, thickness=cv2.FILLED)

contour_vis = img_rgb.copy()
for cnt in valid_contours:
    hull = cv2.convexHull(cnt)
    cv2.drawContours(contour_vis, [cnt], -1, (0, 255, 80), 3)
    cv2.drawContours(contour_vis, [hull], -1, (255, 220, 0), 2)

cutout = cv2.bitwise_and(img_rgb, img_rgb, mask=selected_mask)

fig, axes = plt.subplots(1, 4, figsize=(16, 4.2))
axes[0].imshow(chosen_raw, cmap=chosen_cmap)
axes[0].set_title(f"1. Peta Saliency/Awal")
axes[1].imshow(selected_mask, cmap="gray")
axes[1].set_title("2. Mask Objek Terpilih")
axes[2].imshow(contour_vis)
axes[2].set_title("3. Kontur (Hijau) & Convex Hull (Kuning)")
axes[3].imshow(cutout)
axes[3].set_title("4. Isolasi Foreground (Cutout)")
for ax in axes:
    ax.axis("off")

fig.suptitle(f"Penyempurnaan Segmentasi ({chosen_label}) — {len(valid_contours)} Objek Terdeteksi", fontsize=13, fontweight="bold")
plt.tight_layout()
save_fig(fig, "03_segmentasi_terpilih.png")
plt.show()

## Cell 7 — Tahap 3: Ekstraksi Fitur (Geometri Kontur, HOG, Tekstur GLCM, & Warna)

Mengubah objek kendaraan hasil segmentasi menjadi **vektor/tabel angka** dan **visualisasi fitur**:
1. **Fitur Bentuk / Geometri**: `Area`, `Perimeter`, `x, y, w, h`, `Aspect Ratio (w/h)`, `Extent`, `Solidity`, `Circularity`, dan `Hu Moments`.
2. **Fitur Gradien Bentuk (HOG)**: menangkap pola orientasi tepi bodi, kaca, dan roda kendaraan.
3. **Fitur Tekstur (GLCM)**: `Contrast`, `Dissimilarity`, `Homogeneity`, `Energy`, `Correlation`.
4. **Fitur Warna (RGB/HSV)**: dicatat sebagai pembanding (kurang diskriminatif untuk jenis kendaraan karena warna kendaraan sangat bervariasi).

In [ ]:
def extract_all_features(rgb: np.ndarray, gray: np.ndarray, mask: np.ndarray, cnt: np.ndarray, obj_id: int = 1):
    area = float(cv2.contourArea(cnt))
    perimeter = float(cv2.arcLength(cnt, True))
    x, y, w, h = cv2.boundingRect(cnt)
    bbox_area = float(max(1, w * h))
    aspect_ratio = w / float(max(1, h))
    extent = area / bbox_area
    hull = cv2.convexHull(cnt)
    hull_area = float(max(1.0, cv2.contourArea(hull)))
    solidity = area / hull_area
    circularity = (4.0 * math.pi * area) / (perimeter ** 2) if perimeter > 0 else 0.0

    hu = cv2.HuMoments(cv2.moments(cnt)).flatten()
    hu_log = [-math.copysign(1.0, val) * math.log10(abs(val)) if abs(val) > 1e-12 else 0.0 for val in hu[:3]]

    roi_rgb = rgb[y : y + h, x : x + w]
    roi_gray = gray[y : y + h, x : x + w]
    roi_mask = mask[y : y + h, x : x + w]

    roi_128 = cv2.resize(roi_gray, (128, 128), interpolation=cv2.INTER_AREA)
    hog_vec, hog_vis = hog(
        roi_128,
        orientations=9,
        pixels_per_cell=(16, 16),
        cells_per_block=(2, 2),
        block_norm="L2-Hys",
        visualize=True,
    )

    roi_q = (roi_128 // 8).astype(np.uint8)
    glcm = graycomatrix(
        roi_q,
        distances=[1, 2],
        angles=[0, np.pi / 4, np.pi / 2, 3 * np.pi / 4],
        levels=32,
        symmetric=True,
        normed=True,
    )
    glcm_props = {
        prop: float(np.mean(graycoprops(glcm, prop)))
        for prop in ("contrast", "dissimilarity", "homogeneity", "energy", "correlation")
    }

    fg_rgb = roi_rgb[roi_mask > 0]
    if fg_rgb.size == 0:
        fg_rgb = roi_rgb.reshape(-1, 3)
    hsv_roi = cv2.cvtColor(roi_rgb, cv2.COLOR_RGB2HSV)
    fg_hsv = hsv_roi[roi_mask > 0]
    if fg_hsv.size == 0:
        fg_hsv = hsv_roi.reshape(-1, 3)

    row = {
        "Objek": obj_id,
        "BBox_x": x,
        "BBox_y": y,
        "BBox_w": w,
        "BBox_h": h,
        "Area_px2": round(area, 1),
        "Perimeter_px": round(perimeter, 1),
        "Aspect_Ratio": round(aspect_ratio, 3),
        "Extent": round(extent, 3),
        "Solidity": round(solidity, 3),
        "Circularity": round(circularity, 3),
        "Hu1_log": round(hu_log[0], 3),
        "Hu2_log": round(hu_log[1], 3),
        "HOG_Dimensi": int(hog_vec.size),
        "GLCM_Contrast": round(glcm_props["contrast"], 3),
        "GLCM_Dissimilarity": round(glcm_props["dissimilarity"], 3),
        "GLCM_Homogeneity": round(glcm_props["homogeneity"], 3),
        "GLCM_Energy": round(glcm_props["energy"], 3),
        "GLCM_Correlation": round(glcm_props["correlation"], 3),
        "RGB_Mean": f"({fg_rgb[:,0].mean():.0f}, {fg_rgb[:,1].mean():.0f}, {fg_rgb[:,2].mean():.0f})",
        "HSV_Sat_Mean": round(float(fg_hsv[:, 1].mean()), 1),
    }
    artifacts = {
        "bbox": (x, y, w, h),
        "roi_rgb": roi_rgb,
        "roi_128": roi_128,
        "hog_vec": hog_vec,
        "hog_vis": hog_vis,
        "glcm_props": glcm_props,
        "fg_rgb": fg_rgb,
    }
    return row, artifacts


feature_rows = []
feature_artifacts = []
for idx, cnt in enumerate(valid_contours, start=1):
    row, art = extract_all_features(img_rgb, prep["enhanced"], selected_mask, cnt, obj_id=idx)
    feature_rows.append(row)
    feature_artifacts.append(art)

df_features = pd.DataFrame(feature_rows)
df_features.to_csv(OUT_DIR / "04_tabel_fitur.csv", index=False)

print("=== TABEL HASIL EKSTRAKSI FITUR (BENTUK, HOG, GLCM, WARNA) ===")
display(df_features.T.rename(columns=lambda c: f"Objek #{c + 1}"))

art0 = feature_artifacts[0]
print(f"\nDimensi vektor fitur HOG : {art0['hog_vec'].size} angka")
print("10 nilai pertama vektor HOG:", np.round(art0["hog_vec"][:10], 4))

fig = plt.figure(figsize=(16, 8.2))
gs = fig.add_gridspec(2, 3, hspace=0.35, wspace=0.22)

ax1 = fig.add_subplot(gs[0, 0])
ax1.imshow(art0["roi_rgb"])
bx, by, bw, bh = art0["bbox"]
ax1.set_title(f"1. ROI Objek Hasil Segmentasi\n({bw}x{bh} px | AR={feature_rows[0]['Aspect_Ratio']})")
ax1.axis("off")

ax2 = fig.add_subplot(gs[0, 1])
ax2.imshow(art0["hog_vis"], cmap="inferno")
ax2.set_title(f"2. Visualisasi Fitur HOG\n(Vektor = {art0['hog_vec'].size} dimensi)")
ax2.axis("off")

ax3 = fig.add_subplot(gs[0, 2])
geom_labels = ["Aspect\nRatio", "Extent", "Solidity", "Circularity"]
geom_vals = [
    feature_rows[0]["Aspect_Ratio"],
    feature_rows[0]["Extent"],
    feature_rows[0]["Solidity"],
    feature_rows[0]["Circularity"],
]
bars = ax3.bar(geom_labels, geom_vals, color=["#4c72b0", "#55a868", "#c44e52", "#8172b2"])
for bar, val in zip(bars, geom_vals):
    ax3.text(bar.get_x() + bar.get_width() / 2, val + 0.03, f"{val:.2f}", ha="center", fontsize=9, fontweight="bold")
ax3.set_title("3. Fitur Geometri / Bentuk")
ax3.set_ylim(0, max(geom_vals) * 1.25 + 0.1)
ax3.grid(axis="y", alpha=0.25)

ax4 = fig.add_subplot(gs[1, 0:2])
glcm_keys = list(art0["glcm_props"].keys())
glcm_vals = [art0["glcm_props"][k] for k in glcm_keys]
bars_g = ax4.barh(glcm_keys, glcm_vals, color="#dd8452")
for bar, val in zip(bars_g, glcm_vals):
    ax4.text(val + max(glcm_vals) * 0.02, bar.get_y() + bar.get_height() / 2, f"{val:.3f}", va="center", fontsize=9)
ax4.set_title("4. Fitur Tekstur GLCM (Gray-Level Co-occurrence Matrix)")
ax4.set_xlim(0, max(glcm_vals) * 1.22 + 0.1)
ax4.grid(axis="x", alpha=0.25)

ax5 = fig.add_subplot(gs[1, 2])
for ch, col, lbl in zip(range(3), ["red", "green", "blue"], ["R", "G", "B"]):
    ax5.hist(art0["fg_rgb"][:, ch], bins=32, range=(0, 256), color=col, alpha=0.45, label=lbl)
ax5.set_title("5. Histogram Warna RGB Foreground\n(Kurang diskriminatif utk jenis kendaraan)")
ax5.legend()
ax5.grid(alpha=0.25)

fig.suptitle(f"Tahap 3: Ekstraksi Fitur PCV Klasik — {active_name}", fontsize=14, fontweight="bold")
save_fig(fig, "04_ekstraksi_fitur.png")
plt.show()

## Cell 8 — Tahap 4: Hasil Akhir Bounding Box Manual (Tanpa Label) & Diagram Alur 1-Baris (Siap PPT)

Menampilkan **Bounding Box manual tanpa label kelas** (`cv2.boundingRect`) dan **ringkasan alur 5-panel horizontal** siap tempel ke slide PowerPoint.

In [ ]:
SHOW_SIZE_TEXT = False  # Set True jika ingin menampilkan ukuran piksel (tetap tanpa label kelas)

bbox_vis = img_rgb.copy()
for art in feature_artifacts:
    x, y, w, h = art["bbox"]
    cv2.rectangle(bbox_vis, (x, y), (x + w, y + h), (40, 220, 80), 4)
    if SHOW_SIZE_TEXT:
        cv2.putText(
            bbox_vis, f"{w}x{h} px", (x + 4, max(22, y - 8)),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (40, 220, 80), 2, cv2.LINE_AA,
        )

fig1, ax = plt.subplots(figsize=(7.5, 6))
ax.imshow(bbox_vis)
ax.set_title(f"Hasil Akhir PCV Klasik: Bounding Box Manual (Tanpa Label)\nMetode: {chosen_label}")
ax.axis("off")
save_fig(fig1, "05_hasil_bounding_box.png")
plt.show()

fig2, axes = plt.subplots(1, 5, figsize=(20, 4.3))
summary_panels = [
    ("1. Input Citra", img_rgb, None),
    (f"2. Preprocessing\n(Blur={rule_cfg['gaussian_kernel']}, γ={rule_cfg['gamma']})", prep["enhanced"], "gray"),
    (f"3. Segmentasi\n(Saliency + GrabCut)", selected_mask, "gray"),
    ("4. Ekstraksi Fitur\n(HOG Gradien ROI)", feature_artifacts[0]["hog_vis"], "inferno"),
    ("5. Output Akhir\n(Bounding Box Manual)", bbox_vis, None),
]
for ax, (title, im, cmap) in zip(axes, summary_panels):
    ax.imshow(im, cmap=cmap)
    ax.set_title(title, fontsize=10.5)
    ax.axis("off")

fig2.suptitle(f"Ringkasan Alur PCV Klasik Adaptif — {active_name[:36]}", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.90])
save_fig(fig2, "06_alur_lengkap_pcv_klasik.png")
plt.show()

## Cell 9 — Evaluasi Semua Gambar Uji (Adaptif `if-else` per Gambar) & Unduh Paket Gambar PPT

Menjalankan seluruh alur PCV klasik adaptif pada semua gambar uji (misal 4 gambar `data/helper`), menyimpan tabel ringkasan CSV, grafik multi-gambar PNG 300 DPI, dan mengemas semua output ke `nb1_pcv_klasik_outputs.zip`.

In [ ]:
rows_n = len(loaded_images)
fig, axes = plt.subplots(rows_n, 4, figsize=(16.5, 3.8 * rows_n))
if rows_n == 1:
    axes = np.expand_dims(axes, axis=0)

multi_summary = []
for r, (name, rgb_i) in enumerate(loaded_images):
    cfg_i = analyze_image_and_choose_rules(rgb_i, auto=AUTO_ADAPTIVE, manual_cfg=MANUAL_PARAMS)
    p_i = preprocess_classic(rgb_i, cfg_i)
    seg_i, key_i = run_all_classic_segmentations(p_i, cfg_i)
    if OVERRIDE_METHOD in seg_i:
        key_i = OVERRIDE_METHOD
    s_i = seg_i[key_i]["mask"]
    sal_i = seg_i["saliency_grabcut"]["raw"]
    cnts_i = extract_valid_contours(s_i, MIN_AREA_RATIO, MAX_AREA_RATIO, KEEP_TOP_K_OBJECTS)
    mask_i = np.zeros_like(s_i)
    vis_i = rgb_i.copy()
    if cnts_i:
        cv2.drawContours(mask_i, cnts_i, -1, 255, thickness=cv2.FILLED)
        for cnt in cnts_i:
            x, y, w, h = cv2.boundingRect(cnt)
            cv2.rectangle(vis_i, (x, y), (x + w, y + h), (40, 220, 80), 4)
        bx, by, bw, bh = cv2.boundingRect(cnts_i[0])
        ar = round(bw / float(max(1, bh)), 2)
    else:
        bx, by, bw, bh, ar = 0, 0, 0, 0, 0.0

    multi_summary.append({
        "Gambar": name[:24],
        "Letterbox": cfg_i["stats"]["letterbox"],
        "Silau_%": cfg_i["stats"]["bright_ratio_%"],
        "Strategi_if_else": cfg_i["grabcut_strategy"],
        "BBox_xywh": f"({bx},{by}) {bw}x{bh}",
        "Aspect_Ratio": ar,
        "Cakupan_%": round(100.0 * np.mean(mask_i > 0), 1),
    })

    axes[r, 0].imshow(rgb_i)
    axes[r, 0].set_title(f"Input: {name[:22]}\n(silau={cfg_i['stats']['bright_ratio_%']}%)", fontsize=9)
    axes[r, 1].imshow(sal_i, cmap="inferno")
    axes[r, 1].set_title("Peta Saliency (Lab+FFT)", fontsize=9)
    axes[r, 2].imshow(make_overlay(rgb_i, mask_i))
    axes[r, 2].set_title(f"Mask ({cfg_i['grabcut_strategy']})\narea={100.0*np.mean(mask_i>0):.1f}%", fontsize=9)
    axes[r, 3].imshow(vis_i)
    axes[r, 3].set_title(f"Bounding Box Manual\n({bx},{by}) {bw}x{bh}", fontsize=9)
    for c in range(4):
        axes[r, c].axis("off")

fig.suptitle("Evaluasi Multi-Gambar: PCV Klasik Adaptif (Saliency + GrabCut if-else)", fontsize=14, fontweight="bold")
plt.tight_layout(rect=[0, 0, 1, 0.96])
save_fig(fig, "07_evaluasi_multi_gambar_adaptif.png")
plt.show()

df_multi = pd.DataFrame(multi_summary)
df_multi.to_csv(OUT_DIR / "07_ringkasan_aturan_if_else.csv", index=False)
display(df_multi)

zip_path = OUT_DIR.parent / "nb1_pcv_klasik_outputs.zip"
with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(OUT_DIR.glob("*")):
        if p.is_file():
            zf.write(p, p.name)
print(f"\nSemua gambar PPT tersimpan di: {zip_path} ({zip_path.stat().st_size / 1024:.1f} KB)")
# Hapus tanda '#' di bawah jika ingin langsung mengunduh ZIP di Colab:
# if IN_COLAB:
#     colab_files.download(str(zip_path))